In [31]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])
SEED = [0, 1, 2,]

In [32]:
"""
Trade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).

Berücksichtigt nur:
  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order
  - Transaktionskosten: geschätzter Spread in Prozent

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv


class TradeRepublicStockTradingEnv(StockTradingEnv):
    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):
        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis
        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %

        # Prozentuale Kosten des Basis-Envs abschalten
        n = kwargs["stock_dim"]
        kwargs["buy_cost_pct"] = [0.0] * n
        kwargs["sell_cost_pct"] = [0.0] * n
        super().__init__(df=df, **kwargs)

    def _turbulent(self):
        return (self.turbulence_threshold is not None
                and self.turbulence >= self.turbulence_threshold)

    def _sell_stock(self, index, action):
        price = self.state[1 + index]
        held = self.state[1 + self.stock_dim + index]
        if price <= 0 or held <= 0:
            return 0

        # Bei Turbulenz alles verkaufen (wie im Basis-Env)
        shares = int(held if self._turbulent() else min(abs(action), held))
        proceeds = price * shares * (1 - self.spread_pct)
        if shares <= 0 or proceeds <= self.order_fee:
            return 0

        self.state[0] += proceeds - self.order_fee
        self.state[1 + self.stock_dim + index] -= shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

    def _buy_stock(self, index, action):
        price = self.state[1 + index]
        if price <= 0 or self._turbulent():
            return 0

        ask = price * (1 + self.spread_pct)
        affordable = int((self.state[0] - self.order_fee) // ask)
        shares = int(min(affordable, action))
        if shares <= 0:
            return 0

        self.state[0] -= ask * shares + self.order_fee
        self.state[1 + self.stock_dim + index] += shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares  """

'\nTrade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).\n\nBerücksichtigt nur:\n  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order\n  - Transaktionskosten: geschätzter Spread in Prozent\n\nfrom finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv\n\n\nclass TradeRepublicStockTradingEnv(StockTradingEnv):\n    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):\n        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis\n        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %\n\n        # Prozentuale Kosten des Basis-Envs abschalten\n        n = kwargs["stock_dim"]\n        kwargs["buy_cost_pct"] = [0.0] * n\n        kwargs["sell_cost_pct"] = [0.0] * n\n        super().__init__(df=df, **kwargs)\n\n    def _turbulent(self):\n        return (self.turbulence_threshold is not None\n                and self.turbulence >= self.turbulence_threshold)\n\n    def _sell_stock(self, index, action)

In [33]:
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv

class TRStockTradingEnv(StockTradingEnv):
    """StockTradingEnv mit Trade-Republic-Fixgebuehr von 1 EUR je Ausfuehrung.

    Die geerbten buy_cost_pct / sell_cost_pct bleiben als Spread aktiv.
    Quelle der Kondition: Preis- und Leistungsverzeichnis Trade Republic.
    """

    def __init__(self, *args, fixed_fee=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_fee = fixed_fee
        self.fee_total = 0.0

    def _charge_fee(self, shares):
        if shares > 0:
            self.state[0] -= self.fixed_fee   # Cash belasten
            self.cost += self.fixed_fee       # in FinRLs Kostenzaehler
            self.fee_total += self.fixed_fee  # eigener Zaehler fuer die Auswertung

    def _buy_stock(self, index, action):
        shares = super()._buy_stock(index, action)
        self._charge_fee(shares)
        return shares

    def _sell_stock(self, index, action):
        shares = super()._sell_stock(index, action)
        self._charge_fee(shares)
        return shares

    def reset(self, *, seed=None, options=None):
        out = super().reset(seed=seed, options=options)
        self.fee_total = 0.0
        return out

In [34]:
#Load Data from 01_data_ops_notebook
train = pd.read_csv('../../data/train.csv')
date = train.columns[0]
train.index = pd.factorize(train[date])[0]
train.index.names = ['']
train.head()


,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
,,,,,,,,,,,,,,,,,,
0,2009-01-02,ADS.DE,22.034235,22.224390,20.996305,21.075537,580197.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,22.034235,22.034235,39.189999,0.0
0,2009-01-02,ALV.DE,32.786087,33.036815,31.957402,32.726592,1424010.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,32.786087,32.786087,39.189999,0.0
0,2009-01-02,BAS.DE,11.868842,11.953924,11.643377,11.868842,3752004.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,11.868842,11.868842,39.189999,0.0
0,2009-01-02,BAYN.DE,26.211126,26.253964,25.678703,25.996931,3088175.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,26.211126,26.211126,39.189999,0.0
0,2009-01-02,BEI.DE,34.603619,35.324017,34.505384,34.726412,356642.0,4.0,0.0,22.140397,21.983533,100.0,66.666667,100.0,34.603619,34.603619,39.189999,0.0


In [35]:
#Setup Environment
stock_dimensions = len(train.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS)*stock_dimensions
print(f"Stock Dimensions:  {stock_dimensions}, State Space: {state_space}")

Stock Dimensions:  31, State Space: 311


In [36]:
buy_cost_list = sell_cost_list = [0.0005] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 50,
    "initial_amount": 100000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
#Setup of the environment with given attributes and data set
e_training_gym = TRStockTradingEnv(df = train, fixed_fee=1.0, **env_tweeks)


In [37]:
#Environment for Training

env_train, _ = e_training_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [38]:
#Setup Agent

agent = DRLAgent(env = env_train)

#Set if_using_MODEL = true if u want to use other algorithms
if_using_a2c = False
if_using_ddpg = False
if_using_ppo = False
if_using_td3 = False
if_using_sac = True

In [39]:
from stable_baselines3.common.callbacks import BaseCallback
class PortfolioCallback(BaseCallback):
    """Loggt Depotwert, Kosten und Orderzahl aus dem Environment."""

    def _on_step(self) -> bool:
        env = self.training_env                       # DummyVecEnv aus get_sb_env()
        self.logger.record("portfolio/asset_value", 
        env.get_attr("asset_memory")[0][-1])
        self.logger.record("portfolio/cost_cum",    env.get_attr("cost")[0])
        self.logger.record("portfolio/trades_cum",  env.get_attr("trades")[0])
        self.logger.record("portfolio/fee_total",   env.get_attr("fee_total")[0])
        return True

In [40]:
from collections.abc import Set
from turtle import setup


if if_using_a2c:
    for seed in SEED:
        #Using model A2C
        model_a2c =agent.get_model("a2c", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/a2c' + f'/seed_{seed}'
        new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_a2c.set_logger(new_logger_a2c)

        trained_a2c = agent.train_model(
                model = model_a2c,
                tb_log_name = 'a2c',
                total_timesteps = 200_000, #Increase for more learning
                callbacks = [PortfolioCallback()]
            )

        trained_a2c.save(f"{TRAINED_MODEL_DIR}/agent_a2c_seed_{seed}")

if if_using_ddpg:
    for seed in SEED:
        #Using model DDPG
        model_ddpg =agent.get_model("ddpg", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/ddpg' + f'/seed_{seed}'
        new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_ddpg.set_logger(new_logger_ddpg)

        trained_ddpg = agent.train_model(
                model = model_ddpg,
                tb_log_name = 'ddpg',
                total_timesteps = 200_000, #Increase for more learning
                callbacks = [PortfolioCallback()]
            )

        trained_ddpg.save(f"{TRAINED_MODEL_DIR}/agent_ddpg_seed_{seed}")

if if_using_ppo:
    for seed in SEED:
        #Using model PPO
        model_ppo =agent.get_model("ppo", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/ppo' + f'/seed_{seed}'
        new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_ppo.set_logger(new_logger_ppo)

        trained_ppo = agent.train_model(
                model = model_ppo,
                tb_log_name = 'ppo',
                total_timesteps = 200_000, #Increase for more learning
                callbacks = [PortfolioCallback()]
            )
        trained_ppo.save(f"{TRAINED_MODEL_DIR}/agent_ppo_seed_{seed}")


if if_using_td3:
    for seed in SEED:
        #Using model TD3
        model_td3 =agent.get_model("td3", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/td3' + f'/seed_{seed}'
        new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_td3.set_logger(new_logger_td3)

#if if_using_sac:
    for seed in SEED:
        #Using model SAC
        model_sac =agent.get_model("sac", seed=seed)
        #setup new logger
        tmp_path = RESULTS_DIR + '/sac' + f'/seed_{seed}'
        new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
        #Set new logger
        model_sac.set_logger(new_logger_sac)

        trained_sac = agent.train_model(
                model = model_sac,
                tb_log_name = 'sac',
                total_timesteps = 200_000, #Increase for more learning
                callbacks = [PortfolioCallback()]
            )
        trained_sac.save(f"{TRAINED_MODEL_DIR}/agent_sac_seed_{seed}")


if if_using_sac:
    for seed in SEED:
        model_sac = agent.get_model("sac", seed=seed)

        tmp_path = RESULTS_DIR + "/sac" + f"/seed_{seed}"
        new_logger_sac = configure(
            tmp_path, ["stdout", "csv", "tensorboard"]
        )
        model_sac.set_logger(new_logger_sac)

        trained_sac = model_sac.learn(
            total_timesteps=200_000,
            tb_log_name="sac",
            callback=PortfolioCallback(),  # weglassen, falls kein eigener Callback gebraucht wird
        )

        trained_sac.save(f"{TRAINED_MODEL_DIR}/agent_sac_seed_{seed}")

{'batch_size': 64, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device
Logging to results/sac/seed_0
---------------------------------
| portfolio/         |          |
|    asset_value     | 1e+05    |
|    cost_cum        | 0        |
|    fee_total       | 0        |
|    trades_cum      | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 72       |
|    time_elapsed    | 159      |
|    total_timesteps | 11536    |
| train/             |          |
|    actor_loss      | -4.86    |
|    critic_loss     | 0.464    |
|    ent_coef        | 0.0377   |
|    ent_coef_loss   | -168     |
|    learning_rate   | 0.0001   |
|    n_updates       | 11435    |
---------------------------------
---------------------------------
| portfolio/         |          |
|    asset_value     | 1e+05    |
|    cost_cum        | 0        |
|    fee_total       | 0        |
|    trades_cum      |

In [41]:
# Portfolio am Ende des Trainingszeitraums auswerten
if if_using_a2c:
    portfolio_model = trained_a2c
elif if_using_ddpg:
    portfolio_model = trained_ddpg
elif if_using_ppo:
    portfolio_model = trained_ppo
elif if_using_sac:
    portfolio_model = trained_sac
else:
    raise ValueError("Aktiviere mindestens ein trainiertes Modell.")

portfolio_env = TRStockTradingEnv(df=train, **env_tweeks)
reset_result = portfolio_env.reset()
obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result

for _ in range(len(train.index.unique())):
    action, _ = portfolio_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = portfolio_env.step(action)
    if terminated or truncated:
        break

stock_count = portfolio_env.stock_dim
state = portfolio_env.state
cash = float(state[0])
prices = state[1:1 + stock_count]
shares = state[1 + stock_count:1 + 2 * stock_count]

ticker_order = train.tic.drop_duplicates().tolist()
portfolio = pd.DataFrame({
    "Aktie": ticker_order,
    "Stueck": shares,
    "Kurs_EUR": prices,
})
portfolio["Wert_EUR"] = portfolio["Stueck"] * portfolio["Kurs_EUR"]
total_value = cash + portfolio["Wert_EUR"].sum()
portfolio["Anteil_Prozent"] = portfolio["Wert_EUR"] / total_value * 100
portfolio = portfolio.loc[portfolio["Stueck"] > 0].sort_values("Wert_EUR", ascending=False)

print(f"Barbestand: {cash:.2f} EUR | Gesamtportfoliowert: {total_value:.2f} EUR")
if portfolio.empty:
    print("Am Ende des Zeitraums wurden keine Aktien gehalten.")
else:
    display(portfolio.reset_index(drop=True).round({"Kurs_EUR": 2, "Wert_EUR": 2, "Anteil_Prozent": 2}))

Barbestand: 2.27 EUR | Gesamtportfoliowert: 481464.88 EUR


,Aktie,Stueck,Kurs_EUR,Wert_EUR,Anteil_Prozent
0,HEN3.DE,6359.0,72.65,461997.47,95.96
1,EOAN.DE,1872.0,7.74,14494.69,3.01
2,MBG.DE,210.0,23.73,4983.86,1.04
3,SAP.DE,7.0,125.03,875.19,0.18
4,HNR1.DE,7.0,115.20,806.38,0.17
5,RHM.DE,5.0,71.26,356.32,0.07
6,SIE.DE,2.0,89.27,178.55,0.04
7,DBK.DE,21.0,6.94,145.76,0.03
